# Random forest — churn

This notebook is one of three model notebooks. The other two are logistic regression and the gradient-boosted trees.
The team uses the same Parquet table, the same seed (42) and the same stratified 75/25 split, so the three results can be compared later in `02d_compare_models.ipynb`.

Put your name on the line below and commit this file from your own git account.

Name:

A random forest grows many trees on bootstrap samples and lets them vote. Each tree also sees a random subset of the columns. One deep tree memorises people. Averaging many shallow trees is the point of this notebook. Features are not standardised: a split is "spend greater than X", and scaling would not change that split.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyspark
import seaborn as sns
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)

In [ ]:
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed" / "customer_churn.parquet").exists():
            return candidate
        if (candidate / "PerfectPetClub.db").exists():
            return candidate
    raise FileNotFoundError("Could not find the project folder. Run the ETL notebook first.")


def ensure_java() -> None:
    if os.environ.get("JAVA_HOME"):
        return
    candidates = sorted(Path(r"C:\Program Files\Microsoft").glob("jdk-17*"))
    if not candidates:
        candidates = sorted(Path(r"C:\Program Files\Java").glob("jdk-17*"))
    if not candidates:
        raise RuntimeError("JAVA_HOME is not set and JDK 17 was not found. PySpark needs a JDK.")
    os.environ["JAVA_HOME"] = str(candidates[-1])
    os.environ["PATH"] = str(candidates[-1] / "bin") + os.pathsep + os.environ.get("PATH", "")


def ensure_hadoop() -> None:
    hadoop_home = PROJECT_ROOT / "tools" / "hadoop"
    winutils = hadoop_home / "bin" / "winutils.exe"
    if not winutils.exists():
        print("tools/hadoop/bin/winutils.exe is not present. Spark will fall back if the Parquet read fails.")
        return
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["PATH"] = str(hadoop_home / "bin") + os.pathsep + os.environ.get("PATH", "")


PROJECT_ROOT = find_project_root()
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "customer_churn.parquet"
CONTRACT_PATH = PROJECT_ROOT / "data" / "processed" / "feature_contract.json"
RESULT_PATH = PROJECT_ROOT / "data" / "processed" / "model_results" / "random_forest.json"
SEED = 42
TEST_FRACTION = 0.25
MODEL_NAME = "random_forest"
sns.set_theme(style="whitegrid")

ensure_java()
ensure_hadoop()
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_HOME"] = str(Path(pyspark.__file__).resolve().parent)

spark_local = Path(tempfile.gettempdir()) / "perfectpet-spark"
(spark_local / "warehouse").mkdir(parents=True, exist_ok=True)
(spark_local / "local").mkdir(parents=True, exist_ok=True)

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder.master("local[2]")
    .appName("perfectpetclub-forest")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.sql.warehouse.dir", (spark_local / "warehouse").as_posix())
    .config("spark.local.dir", (spark_local / "local").as_posix())
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)
print("Model notebook:", MODEL_NAME)

## Load

The column list comes from `feature_contract.json`, written by the ETL notebook. Customer id is not a feature. A column with only one value is dropped, because it cannot separate customers and it breaks standardisation in the logistic notebook.

In [ ]:
def load_customers(parquet_path: Path):
    uri = parquet_path.resolve().as_uri()
    try:
        frame = spark.read.parquet(uri)
        frame.limit(1).collect()
        print("Loaded Parquet with Spark.")
        return frame
    except Exception as error:
        print(f"Spark could not read the Parquet path directly ({type(error).__name__}). Loading via pyarrow.")
        return spark.createDataFrame(pd.read_parquet(parquet_path))


if not PARQUET_PATH.exists() or not CONTRACT_PATH.exists():
    raise FileNotFoundError("Run the ETL notebook first so customer_churn.parquet and feature_contract.json exist.")

contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
customers = load_customers(PARQUET_PATH)
id_column = contract["id_column"]
label_column = contract["label_column"]
categorical_features = list(contract["categorical_features"])
numeric_features = list(contract["numeric_features"])

missing = [column for column in [id_column, label_column, *categorical_features, *numeric_features] if column not in customers.columns]
if missing:
    raise RuntimeError(f"Parquet file is missing contract columns: {missing}")
if customers.count() != contract["row_count"]:
    raise RuntimeError(f"Parquet has {customers.count()} rows but the contract says {contract['row_count']}.")

print(contract["positive_class"])
print("Snapshot", contract["as_of"], "  rows", contract["row_count"], "  churn rate", round(contract["churn_rate"], 3))

varying_numeric = []
for column in numeric_features:
    distinct_values = customers.where(F.col(column).isNotNull()).select(column).distinct().count()
    if distinct_values <= 1:
        print(f"Dropping {column}: {distinct_values} non-null value, so it cannot separate customers.")
    else:
        varying_numeric.append(column)
numeric_features = varying_numeric

varying_categorical = []
for column in categorical_features:
    distinct_values = customers.select(column).distinct().count()
    if distinct_values < 2:
        print(f"Dropping {column}: only {distinct_values} level.")
    else:
        varying_categorical.append(column)
categorical_features = varying_categorical
customers.printSchema()

## A query to keep handy

If a marker asks for a mean, a filter or a sort, edit this cell. It does not change the model.

In [ ]:
customers.groupBy("subscription_type").agg(
    F.count("*").alias("customers"),
    F.round(F.mean("total_spend"), 1).alias("mean_spend"),
    F.round(F.mean(label_column), 3).alias("churn_rate"),
).orderBy(F.desc("churn_rate")).show()

## Split

The same seed and the same stratified split are used in all three model notebooks. `randomSplit` on the whole table can put almost every churner on one side, so each class is split on its own. The weight is `n / (2 * n_class)` on the training fold.

In [ ]:
def stratified_split(frame, label: str, test_fraction: float, seed: int):
    train_parts = []
    test_parts = []
    for value in (0, 1):
        subset = frame.filter(F.col(label) == value)
        train_part, test_part = subset.randomSplit([1 - test_fraction, test_fraction], seed=seed)
        train_parts.append(train_part)
        test_parts.append(test_part)
    return train_parts[0].unionByName(train_parts[1]), test_parts[0].unionByName(test_parts[1])


def add_class_weight(frame, label: str):
    counts = {int(row[label]): int(row["count"]) for row in frame.groupBy(label).count().collect()}
    total = sum(counts.values())
    weights = {value: total / (len(counts) * count) for value, count in counts.items()}
    weighted = frame.withColumn(
        "class_weight",
        F.when(F.col(label) == 1, F.lit(float(weights[1]))).otherwise(F.lit(float(weights[0]))),
    )
    return weighted, weights


train, test = stratified_split(customers, label_column, TEST_FRACTION, SEED)
train, class_weights = add_class_weight(train, label_column)
test = test.withColumn("class_weight", F.lit(1.0))
train = train.cache()
test = test.cache()
print("Training class weights:", class_weights)
train.groupBy(label_column).count().orderBy(label_column).show()
test.groupBy(label_column).count().orderBy(label_column).show()

## Shared preparation

These stages are fit inside each cross-validation fold, not on the holdout. Categories are indexed and one-hot encoded, with one level dropped as the reference. Numeric gaps are filled with the training-fold median.

In [ ]:
def shared_stages(numeric_cols: list[str], categorical_cols: list[str]):
    """Index, one-hot, median-impute, then assemble. Fit inside each cross-validation fold."""
    indexers = [
        StringIndexer(inputCol=column, outputCol=f"{column}_idx", handleInvalid="keep", stringOrderType="alphabetAsc")
        for column in categorical_cols
    ]
    encoder = OneHotEncoder(
        inputCols=[f"{column}_idx" for column in categorical_cols],
        outputCols=[f"{column}_oh" for column in categorical_cols],
        dropLast=True,
        handleInvalid="keep",
    )
    imputed = [f"{column}_imp" for column in numeric_cols]
    imputer = Imputer(inputCols=numeric_cols, outputCols=imputed, strategy="median")
    assembler = VectorAssembler(
        inputCols=imputed + [f"{column}_oh" for column in categorical_cols],
        outputCol="features",
        handleInvalid="keep",
    )
    return indexers + [encoder, imputer, assembler]


auc_evaluator = BinaryClassificationEvaluator(
    labelCol=label_column,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

## This model

Class weights are on, so a churned customer counts for more when the classes are unequal. `minInstancesPerNode=5` means a leaf must cover at least five training customers, so a leaf cannot be one person. Each tree sees 80% of the training rows. The grid is the number of trees (50 or 100) and the depth (3 or 6). Three-fold cross-validation on the training customers picks the setting.

In [ ]:
classifier = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_column,
    weightCol="class_weight",
    seed=SEED,
    minInstancesPerNode=5,
    subsamplingRate=0.8,
)
pipeline = Pipeline(stages=shared_stages(numeric_features, categorical_features) + [classifier])
grid = (
    ParamGridBuilder()
    .addGrid(classifier.numTrees, [50, 100])
    .addGrid(classifier.maxDepth, [3, 6])
    .build()
)

## Fit

Four settings, three folds, scored by ROC AUC. The winning setting is refit on all of the training customers. The test customers are not used to choose it.

In [ ]:
search = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=grid,
    evaluator=auc_evaluator,
    numFolds=3,
    seed=SEED,
    parallelism=1,
)
fitted = search.fit(train)
cv_table = pd.DataFrame({"cv_auc": [float(metric) for metric in fitted.avgMetrics]})
cv_table["selected"] = cv_table["cv_auc"].sub(cv_table["cv_auc"].max()).abs() < 1e-12
display(cv_table.sort_values("cv_auc", ascending=False))
print("Best cross-validated AUC:", round(float(np.max(fitted.avgMetrics)), 3))

## Holdout

Precision and recall are for the churn class. The baseline predicts retained for everyone. Its churn recall is zero. Accuracy near 60% with no recall is that baseline.

In [ ]:
def confusion_counts(frame, label: str) -> dict[str, float]:
    scored = frame.select(label, "prediction").toPandas()
    actual = scored[label].astype(int)
    predicted = scored["prediction"].astype(int)
    tp = int(((actual == 1) & (predicted == 1)).sum())
    tn = int(((actual == 0) & (predicted == 0)).sum())
    fp = int(((actual == 0) & (predicted == 1)).sum())
    fn = int(((actual == 1) & (predicted == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    accuracy = (tp + tn) / len(scored) if len(scored) else 0.0
    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "accuracy": accuracy, "rows": int(len(scored)),
    }


def score_split(model, frame, label: str) -> dict[str, float]:
    predicted = model.transform(frame)
    counts = confusion_counts(predicted, label)
    counts["auc"] = float(auc_evaluator.evaluate(predicted))
    return counts


holdout = score_split(fitted, test, label_column)
training = score_split(fitted, train, label_column)
baseline_frame = test.withColumn("prediction", F.lit(0.0))
baseline = confusion_counts(baseline_frame, label_column)
baseline["auc"] = 0.5

scores = pd.DataFrame([
    {"split": "train", **training},
    {"split": "test", **holdout},
    {"split": "baseline_predict_retained", **baseline},
])
display(scores.round(3))
gap = training["auc"] - holdout["auc"]
print(f"Train AUC minus test AUC: {gap:.3f}")
if gap > 0.1:
    print("The gap is large. The model is fitting the training customers more tightly than the holdout supports.")

In [ ]:
def plot_confusion(counts: dict[str, float], title: str, ax) -> None:
    matrix = np.array([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]], dtype=float)
    sns.heatmap(
        matrix, annot=True, fmt=".0f", cmap="Blues", cbar=False,
        xticklabels=["predicted retained", "predicted churn"],
        yticklabels=["actual retained", "actual churn"],
        ax=ax,
    )
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_confusion(holdout, f"{MODEL_NAME} on holdout", axes[0])
plot_confusion(baseline, "Majority baseline on holdout", axes[1])
fig.tight_layout()

## What the model uses

Importances have no sign. Logistic coefficients do, and they are on the standardised scale. Either way, read them next to the group means in the ETL notebook.

In [ ]:
def expanded_feature_names(pipeline_model, numeric_cols: list[str], categorical_cols: list[str]) -> list[str]:
    names = list(numeric_cols)
    for column in categorical_cols:
        indexer = next(
            stage
            for stage in pipeline_model.stages
            if stage.__class__.__name__ == "StringIndexerModel" and stage.getOutputCol() == f"{column}_idx"
        )
        levels = list(indexer.labels)
        names.extend(f"{column}={level}" for level in levels[:-1])
    return names


def effect_table(pipeline_model, numeric_cols: list[str], categorical_cols: list[str]) -> pd.DataFrame:
    names = expanded_feature_names(pipeline_model, numeric_cols, categorical_cols)
    estimator = pipeline_model.stages[-1]
    if hasattr(estimator, "featureImportances"):
        values = estimator.featureImportances.toArray()
        kind = "importance"
    else:
        values = estimator.coefficients.toArray()
        kind = "coefficient"
    if len(values) > len(names):
        names = names + [f"unseen_or_extra_{index}" for index in range(len(values) - len(names))]
    if len(names) > len(values):
        names = names[: len(values)]
    table = pd.DataFrame({"feature": names, kind: values})
    table["abs_value"] = table[kind].abs()
    return table.sort_values("abs_value", ascending=False).drop(columns="abs_value")


chosen = fitted.bestModel
effects = effect_table(chosen, numeric_features, categorical_features)
display(effects.head(15))

fig, ax = plt.subplots(figsize=(8, 5))
top = effects.head(12).iloc[::-1]
value_column = "importance" if "importance" in top.columns else "coefficient"
sns.barplot(data=top, x=value_column, y="feature", hue="feature", ax=ax, palette="crest", legend=False)
ax.set_title(f"{MODEL_NAME}: largest {value_column}s")
fig.tight_layout()

interesting = {"regParam", "elasticNetParam", "numTrees", "maxDepth", "maxIter", "stepSize", "minInstancesPerNode", "subsamplingRate"}
estimator = chosen.stages[-1]
best_params = {param.name: estimator.getOrDefault(param) for param in estimator.params if param.name in interesting}
print("Chosen settings:", best_params)

## Hand-off

This writes one JSON file. `02d_compare_models.ipynb` reads the three files and puts the models on one chart. Run the three model notebooks before that one.

In [ ]:
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "model": MODEL_NAME,
    "best_cv_auc": float(np.max(fitted.avgMetrics)),
    "cv_aucs": [float(metric) for metric in fitted.avgMetrics],
    "best_params": {key: float(value) if isinstance(value, (float, int)) else value for key, value in best_params.items()},
    "train_auc": training["auc"],
    "test_auc": holdout["auc"],
    "test_accuracy": holdout["accuracy"],
    "test_precision": holdout["precision"],
    "test_recall": holdout["recall"],
    "test_f1": holdout["f1"],
    "baseline_accuracy": baseline["accuracy"],
    "baseline_recall": baseline["recall"],
}
RESULT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", RESULT_PATH)

## What to say about this model

- Importance is the share of split improvement. It has no sign. A large importance on recency means the forest splits on inactivity. It does not say whether more recency raises or lowers churn. The group means in the ETL notebook give the direction: churners last ordered, logged in and called much longer ago.
- Depth 6 with a minimum of five customers per leaf cannot isolate a single person. If the train AUC is still far above the test AUC, say so.
- This Spark random forest accepts `weightCol`. The gradient-boosted tree notebook cannot, in this version of Spark. That difference is a team decision, not an accident.